
# 10-kVA, GFL, converter output admittance identification

This example demonstrates converter output admittance identification using a 10-kVA
grid-following (GFL) converter.


In [ ]:
from motulator.grid import control, model, utils

Compute base values based on the nominal values.



In [ ]:
nom = utils.NominalValues(U=400, I=14.5, f=50, P=10e3)
base = utils.BaseValues.from_nominal(nom)

Configure the identification.



In [ ]:
identification_cfg = utils.IdentificationCfg(
    abs_u_e=0.01 * base.u,
    f_start=1,
    f_stop=10e3,
    n_freqs=100,
    T_s=1 / 10e3,
    # Uncomment the row below to save identification results in "project root"/data
    # filename="gfl_admittance",
    filetype="csv",
)

Configure the system model.



In [ ]:
ac_filter = model.LFilter(L_f=0.2 * base.L)
ac_source = model.ThreePhaseSource(w_g=base.w, e_g=base.u)
converter = model.VoltageSourceConverter(u_dc=650)
mdl = model.GridConverterSystem(converter, ac_filter, ac_source)

Configure the control system.



In [ ]:
cfg = control.CurrentVectorControllerCfg(
    i_max=1.5 * base.i, L=0.2 * base.L, T_s=identification_cfg.T_s
)
inner_ctrl = control.CurrentVectorController(cfg)
ctrl = control.GridConverterControlSystem(inner_ctrl)

# Set the references
ctrl.set_power_ref(0.5 * base.p)
ctrl.set_reactive_power_ref(0.5 * base.p)

Run the identification and plot results.



In [ ]:
if __name__ == "__main__":
    res = utils.run_identification(identification_cfg, mdl, ctrl)
    utils.plot_identification(res, plot_style="re_im")
    utils.plot_vector_identification(res, base)